
# 03 — Feature Engineering (Scaffold, No-Op for This PR)

**Scope of this PR:** Mechanical refit only.  
This notebook now **loads processed artifacts** via `utils/data_io.py` and prepares a clean scaffold for future feature engineering.  
No new features are created in this PR; logic will be added in the next PR.

**This notebook does:**
- Loads `X` and `y` from `data/processed/` (or regenerates from raw via utils if missing)
- Sets up a **no-op** feature engineering pipeline (`X_fe = X.copy()`)
- (Optional) Saves `X_fe` back to `data/processed/` for downstream modeling (identical to `X` for now)


In [1]:

from pathlib import Path
import sys
import pandas as pd

# Ensure Python can import from the repo root (where utils/ lives)
repo_root = Path.cwd()
if not (repo_root / "utils" / "data_io.py").exists():
    repo_root = Path.cwd().parents[0]  # e.g., notebooks/ -> repo root
sys.path.insert(0, str(repo_root))

from utils.data_io import (
    ensure_dirs, load_profile, load_all_sensors,
    build_feature_matrix, save_processed, write_feature_index,
    ROOT_DIR, META_DIR, PROC_DIR
)

ensure_dirs()
print("ROOT_DIR:", ROOT_DIR)
print("PROC_DIR:", PROC_DIR)


ROOT_DIR: C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard
PROC_DIR: C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard\data\processed



## Load processed artifacts (fallback to regeneration)

Loads `data/processed/X_features.parquet` and `y_labels.parquet`.  
If missing, rebuilds from raw using the same logic as notebook **01**.


In [2]:

X_path_pq = PROC_DIR / "X_features.parquet"
y_path_pq = PROC_DIR / "y_labels.parquet"
X_path_csv = PROC_DIR / "X_features.csv"
y_path_csv = PROC_DIR / "y_labels.csv"

def _load_processed_or_rebuild():
    # Try Parquet first
    if X_path_pq.exists() and y_path_pq.exists():
        X = pd.read_parquet(X_path_pq)
        y = pd.read_parquet(y_path_pq)
        print("✅ Loaded processed Parquet artifacts.")
        return X, y
    # Try CSV fallback
    if X_path_csv.exists() and y_path_csv.exists():
        X = pd.read_csv(X_path_csv)
        y = pd.read_csv(y_path_csv)
        print("✅ Loaded processed CSV artifacts.")
        return X, y
    # Rebuild if nothing exists
    print("⚠️ Processed artifacts not found. Rebuilding from raw via utils/data_io.py...")
    labels = load_profile()
    frames, order = load_all_sensors()
    X, y = build_feature_matrix(frames, order, labels)
    save_processed(X, y)
    write_feature_index(X)
    return X, y

X, y = _load_processed_or_rebuild()
print("X shape:", X.shape, "| y shape:", y.shape)


✅ Loaded processed Parquet artifacts.
X shape: (2205, 43680) | y shape: (2205, 5)



## No-op feature engineering scaffold

This is a **placeholder** for future transformations (next PR).  
Examples we may add later: rolling stats, FFT features, aggregation across channels, stability flags, etc.

For this PR, we **do not** alter the data; we simply mirror `X` to `X_fe`.


In [3]:

# Start with a clean copy
X_fe = X.copy()

# --- Placeholder functions (to be implemented in the next PR) ---
def add_rolling_stats(df, window=10):
    """TODO: Add rolling mean/std/min/max per sensor channel."""
    return df

def add_freq_domain_features(df):
    """TODO: Compute simple FFT-based energy/bandpower features per channel."""
    return df

def add_inter_sensor_aggregations(df):
    """TODO: Aggregate by sensor groups (e.g., per-pressure mean/std)."""
    return df

# Example of future pipeline (currently no-op):
# X_fe = add_rolling_stats(X_fe, window=10)
# X_fe = add_freq_domain_features(X_fe)
# X_fe = add_inter_sensor_aggregations(X_fe)

print("X_fe shape:", X_fe.shape, "(no-op, identical to X for this PR)")


X_fe shape: (2205, 43680) (no-op, identical to X for this PR)



## (Optional) Save engineered features

For downstream modeling, some teams prefer a distinct file, even for a no-op.  
We save `X_features_fe.parquet` so the modeling notebook can read a consistent filename across PRs.


In [4]:

out_pq = PROC_DIR / "X_features_fe.parquet"
X_fe.to_parquet(out_pq, index=False)
print("Saved X_fe ->", out_pq)


Saved X_fe -> C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard\data\processed\X_features_fe.parquet



## Summary

- Loaded processed artifacts (`X`, `y`) with automatic fallbacks
- Prepared a **feature engineering scaffold** with placeholder functions
- Saved a **no-op engineered** artifact `X_features_fe.parquet` (identical to `X` for now)

> In the **next PR**, we will implement actual transformations inside these placeholder functions.
